<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/main/notebooks/04_bigru_stacked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stacked BiGRU (Whitespace BPE, validation v2)

Trainable embedding 128 → 3-layer BiGRU h256/direction → BIO logits. Checkpoints are selected only on v2 VAL with viterbi_lines and fixed cleanup. Every epoch is backed up to Drive.


## 1. Setup / install

In [1]:
import os, sys, base64, subprocess
from pathlib import Path
from google.colab import userdata

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    token = userdata.get("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

# Make src importable
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Repository:", repo_dir)


Repository: /content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"

In [3]:
import importlib
for module in ("src.data_io", "src.metrics", "src.training", "src.bigru_helpers", "src.experiment_tracking"):
    importlib.reload(importlib.import_module(module))

import gc, json, platform, uuid
import shutil
from importlib.metadata import version
from time import perf_counter

import pandas as pd
import torch
from tokenizers import Tokenizer
import wandb

from src.bigru_model import StackedBiGRUTagger
from src.bigru_helpers import CLEAN_OPTIONS, backup_files, postprocessed_scores
from src.data_io import get_data_dir, load_manifest, load_data_splits, save_json, save_predictions, file_sha256
from src.dataset import LABEL_TO_ID, prepare_ner_records, prepare_inference_records, make_loader
from src.metrics import competition_scores
from src.preprocessing import clean_locations_v3
from src.training import set_seed, train_epoch, predict, save_checkpoint
from src.experiment_tracking import start_run, log_epoch, finish_run

assert torch.cuda.is_available(), "GPU required for full training: Runtime > Change runtime type > GPU"
device = torch.device("cuda")
print("PyTorch:", torch.__version__, ", GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.14.1+cu130 , GPU: Tesla T4


## 2. External services (W&B + Drive)

In [4]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"
from google.colab import userdata
wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

In [5]:
from google.colab import drive
drive.mount("/content/drive")
checkpoint_backup_root = Path("/content/drive/MyDrive/UA-Locations-NER/stacked_bigru")

print("Checkpoint backup root:", checkpoint_backup_root)


Mounted at /content/drive
Checkpoint backup root: /content/drive/MyDrive/UA-Locations-NER/stacked_bigru


## 3. Control configuration

In [6]:
config = {
    "model": "bigru", "architecture": "stacked", "embedding_dim": 128,
    "hidden_size": 256, "num_layers": 3, "dropout": 0.2,
    "seed": 17, "batch_size": 128, "learning_rate": 0.001, "epochs": 3,
    "gradient_clip": 1.0, "max_padded_tokens": 16384,
    "prepare_batch_size": 2000, "sources": ["uk", "ru"],
    "optimizer": "Adam", "tokenizer": "bpe_whitespace.json",
    "selection_metric": "uk_comp_f1", "label_to_id": LABEL_TO_ID,
    "max_sequence_length": None, "early_stopping": False,
    "environment": ENVIRONMENT, "validation_version": "v2", "manifest_name": "validation_manifest_v2",
    "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "expand_trim_empty",
    "decoding": "viterbi_lines", "clean_options": dict(CLEAN_OPTIONS), "upload_checkpoint": False,
}
set_seed(config["seed"])
splits_dir = repo_dir / "data/splits"
manifest_path = splits_dir / f"{config['manifest_name']}.parquet"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]
run_id = "bigru_ws_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
backup_dir = (checkpoint_backup_root / run_id)
backup_dir.mkdir(parents=True, exist_ok=True)
print(run_id)


bigru_ws_20261118T172717Z_9f8dbe85


In [7]:
provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in
                 ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path),
    "source_sha256": {path.name: file_sha256(path) for path in
                      sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": (
    file_sha256(repo_dir / "notebooks/04_bigru_stacked.ipynb")
    if (repo_dir / "notebooks/04_bigru_stacked.ipynb").is_file()
    else None
),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
backup_files(backup_dir, run_dir/"config.json", run_dir/"provenance.json", tokenizer_path)


## 4. Tokenizer

In [8]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()

pad_id = tokenizer.token_to_id("[PAD]")

print("Whitespace BPE vocabulary:", tokenizer.get_vocab_size())

Whitespace BPE vocabulary: 32000


## 5. Fixed data splits & alignment diagnostics


In [9]:
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir, name=config["manifest_name"])
display(pd.crosstab(manifest["source"], manifest["split"]))

record_sets = {"train": [], "val": [], "holdout": []}
alignment = []
for source in config["sources"]:
    splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,),
                              manifest_name=config["manifest_name"])
    for split, target in record_sets.items():
        frame = splits.pop(f"{source}_{split}")
        records, skipped = prepare_ner_records(
            tokenizer, frame, source, split, batch_size=config["prepare_batch_size"])
        target.extend(records)
        alignment.append({"source": source, "split": split, "total": len(frame),
                          "retained": len(records), "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_skipped.json", skipped)
        del frame, records
    del splits
    gc.collect()
train_records, val_records, holdout_records = (
    record_sets["train"], record_sets["val"], record_sets["holdout"])
for split in ("val", "holdout"):
    expected = manifest[(manifest["split"] == split) & manifest["source"].isin(config["sources"])]
    assert {r["text_id"] for r in record_sets[split]} == {
        f"{row.source}_{row.row_id}" for row in expected.itertuples()}
alignment = pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(run_dir / "alignment.csv", index=False)
backup_files(backup_dir, run_dir / "alignment.csv")

loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": True,
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)
holdout_loader = make_loader(holdout_records, **loader_options)


100%|██████████| 811M/811M [00:47<00:00, 17.7MB/s]

Extracting files...


split,holdout,train,val
source,,,
ru,24743,100000,24626
uk,100757,805983,100741


Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/51 [00:00<?, ?it/s]

Prepare uk/holdout:   0%|          | 0/51 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/13 [00:00<?, ?it/s]

Prepare ru/holdout:   0%|          | 0/13 [00:00<?, ?it/s]

,source,split,total,retained,alignment_failures
0,uk,train,805983,804367,1616
1,uk,val,100741,100741,169
2,uk,holdout,100757,100757,206
3,ru,train,100000,99909,91
4,ru,val,24626,24626,17
5,ru,holdout,24743,24743,21


## 6. Model / W&B run

In [10]:
os.environ["WANDB_DIR"] = str(run_dir)
run = start_run(
    model="bigru", config=config, name=run_id,
    validation_version=config["validation_version"], metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"], repo_dir=repo_dir,
    split_manifest_path=manifest_path,
    tokenizer_path=tokenizer_path, provenance=provenance,
    tags=["stacked", "bigru", "whitespace-bpe", "control"],
)
save_json(run_dir / "wandb_run.json", {
    "id": run.id, "url": run.url, "path": f"{run.entity}/{run.project}/{run.id}",
})
backup_files(backup_dir, run_dir/"wandb_run.json")
print(run.url)


https://app.forge.coreweave.com/daldoshyna-kyiv-school-of-economics/ua-location-ner/runs/m1r3fqua


In [11]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"],
    "num_labels": len(LABEL_TO_ID),
}
set_seed(config["seed"])
model = StackedBiGRUTagger(**model_config).to(device)
optimizer_cls = getattr(torch.optim, config["optimizer"])
optimizer_kwargs = {"lr": config["learning_rate"]}
if config["optimizer"] == "AdamW":
    optimizer_kwargs["weight_decay"] = config.get("weight_decay", 0.01)
optimizer = optimizer_cls(model.parameters(), **optimizer_kwargs)
save_json(run_dir / "model_config.json", model_config)
backup_files(backup_dir, run_dir / "model_config.json")
run.config.update({"model_config": model_config, "optimizer_effective": config["optimizer"]})
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


StackedBiGRUTagger(
  (embedding): Embedding(32000, 128, padding_idx=0)
  (gru): GRU(128, 256, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
)
Trainable parameters: 7055875


## 7. Training on v2 train

Saves every epoch and best.pt locally and to Drive. The same viterbi_lines + cleanup UK VAL criterion


In [12]:
history, best_score, best_epoch = [], -1, None
best_path = run_dir / "best.pt"
for epoch in range(1, config["epochs"] + 1):
    epoch_started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device, decoding=config["decoding"])
    scores = postprocessed_scores(val_records, predictions, config["clean_options"])
    raw_scores = competition_scores(val_records, predictions)
    result = {"epoch": epoch, "train_loss": train_loss,
              "epoch_seconds": perf_counter() - epoch_started,
              **{f"{source}_val_loss": value for source, value in val_losses.items()},
              **scores, **{f"raw_{key}": value for key, value in raw_scores.items()}}
    history.append(result)
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    epoch_path = run_dir / f"epoch_{epoch:02d}.pt"
    save_checkpoint(epoch_path, model, optimizer, epoch, config, model_config, scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score, best_epoch = scores[config["selection_metric"]], epoch
        save_checkpoint(best_path, model, optimizer, epoch, config, model_config, scores, provenance)
    backup_files(backup_dir, epoch_path, best_path, run_dir / "history.csv")
    log_epoch(run, epoch, result, selection_metric=config["selection_metric"],
              epoch_seconds=result["epoch_seconds"],
              extra_metrics={f"val_{config['validation_version']}/raw/{key}": value
                             for key, value in raw_scores.items()})
    print(result)
    del predictions
display(pd.DataFrame(history))
optimizer.zero_grad(set_to_none=True)
del optimizer
gc.collect()


Train:   0%|          | 0/7102 [00:01<?, ?it/s]

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.03665299661632599, 'epoch_seconds': 257.70008145400004, 'uk_val_loss': 0.021060127586596696, 'ru_val_loss': 0.02923291470804586, 'combined_val_loss': 0.0230935775372189, 'uk_comp_f1': 0.85780202, 'ru_comp_f1': 0.86841629, 'combined_comp_f1': 0.86048132, 'precision': 0.8969639242766996, 'recall': 0.8268504697515147, 'TP': 37668, 'FP': 4327, 'FN': 7888, 'raw_uk_comp_f1': 0.85922729, 'raw_ru_comp_f1': 0.8396533, 'raw_combined_comp_f1': 0.85429695, 'raw_precision': 0.8861995753715499, 'raw_recall': 0.824611467205198, 'raw_TP': 37566, 'raw_FP': 4824, 'raw_FN': 7990}


Train:   0%|          | 0/7104 [00:01<?, ?it/s]

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.018185084548656374, 'epoch_seconds': 262.39610379500004, 'uk_val_loss': 0.017968785581246323, 'ru_val_loss': 0.02270094393540837, 'combined_val_loss': 0.01914618161868294, 'uk_comp_f1': 0.86980471, 'ru_comp_f1': 0.89533123, 'combined_comp_f1': 0.87626135, 'precision': 0.873794036757323, 'recall': 0.8787426464132058, 'TP': 40032, 'FP': 5782, 'FN': 5524, 'raw_uk_comp_f1': 0.87440479, 'raw_ru_comp_f1': 0.87802563, 'raw_combined_comp_f1': 0.87531845, 'raw_precision': 0.868341469736899, 'raw_recall': 0.8824084643076653, 'raw_TP': 40199, 'raw_FP': 6095, 'raw_FN': 5357}


Train:   0%|          | 0/7103 [00:01<?, ?it/s]

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.014290259419509236, 'epoch_seconds': 262.65877484199996, 'uk_val_loss': 0.018072847129003304, 'ru_val_loss': 0.02278979121219015, 'combined_val_loss': 0.01924645774316833, 'uk_comp_f1': 0.87233888, 'ru_comp_f1': 0.90327862, 'combined_comp_f1': 0.88015854, 'precision': 0.8732552612246661, 'recall': 0.8871718324699271, 'TP': 40416, 'FP': 5866, 'FN': 5140, 'raw_uk_comp_f1': 0.87701341, 'raw_ru_comp_f1': 0.87999313, 'raw_combined_comp_f1': 0.87776537, 'raw_precision': 0.867004989186527, 'raw_recall': 0.8887962068662745, 'raw_TP': 40490, 'raw_FP': 6211, 'raw_FN': 5066}


,epoch,train_loss,epoch_seconds,uk_val_loss,ru_val_loss,combined_val_loss,uk_comp_f1,ru_comp_f1,combined_comp_f1,precision,...,FP,FN,raw_uk_comp_f1,raw_ru_comp_f1,raw_combined_comp_f1,raw_precision,raw_recall,raw_TP,raw_FP,raw_FN
0,1,0.036653,257.700081,0.021060,0.029233,0.023094,0.857802,0.868416,0.860481,0.896964,...,4327,7888,0.859227,0.839653,0.854297,0.886200,0.824611,37566,4824,7990
1,2,0.018185,262.396104,0.017969,0.022701,0.019146,0.869805,0.895331,0.876261,0.873794,...,5782,5524,0.874405,0.878026,0.875318,0.868341,0.882408,40199,6095,5357
2,3,0.014290,262.658775,0.018073,0.022790,0.019246,0.872339,0.903279,0.880159,0.873255,...,5866,5140,0.877013,0.879993,0.877765,0.867005,0.888796,40490,6211,5066


18

## 8. Load the best VAL checkpoint


In [13]:
checkpoint_path = best_path
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
selected_epoch = checkpoint["epoch"]
summary = {"selected_epoch": selected_epoch, "checkpoint": checkpoint_path.name,
           "selection_metric": config["selection_metric"], "scores": checkpoint["scores"],
           "decoding": config["decoding"], "clean_options": config["clean_options"]}
save_json(run_dir / "summary.json", summary)
backup_files(backup_dir, run_dir / "summary.json")
print(summary)
del checkpoint



{'selected_epoch': 3, 'checkpoint': 'best.pt', 'selection_metric': 'uk_comp_f1', 'scores': {'uk_comp_f1': 0.87233888, 'ru_comp_f1': 0.90327862, 'combined_comp_f1': 0.88015854, 'precision': 0.8732552612246661, 'recall': 0.8871718324699271, 'TP': 40416, 'FP': 5866, 'FN': 5140}, 'decoding': 'viterbi_lines', 'clean_options': {'expand': True, 'trim': True, 'empty': True}}


## 9. Kaggle test submission with selected checkpoint

In [14]:
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
test_loader = make_loader(test_records, **loader_options)
test_predictions, _ = predict(model, test_loader, device,
                               decoding=config["decoding"], return_logits=True)
from src.postprocessing import postprocess
submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(postprocess(row.text, test_predictions[row.text_id]["spans"],
                           **config["clean_options"]), ensure_ascii=False)
    for row in test.itertuples(index=False)
]
submission_path = run_dir / "submission_bigru_viterbi_lines.csv"
submission.to_csv(submission_path, index=False)
save_json(run_dir / "submission_config.json", {
    "checkpoint": str(checkpoint_path), "epoch": selected_epoch,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path), "decoding": config["decoding"],
    "postprocessing": config["clean_options"],
})
backup_files(backup_dir, submission_path, run_dir / "submission_config.json")
display(submission.head())


Predict:   0%|          | 0/6 [00:00<?, ?it/s]

,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Шулявського шляхопроводу"", ""Києва"",..."
2,2,"[""Миргородського району"", ""Полтавської області""]"
3,3,[]
4,4,"[""проспекті Академіка Глушкова""]"


## 10. Export VAL, holdout and TEST with logits
Raw predictions and logits are saved before cleanup to MyDrive. TEST predictions are reused from the submission cell above.


In [15]:
export_dir = Path("/content/drive/MyDrive/ner_results") / config["model"] / (
    f"{run_id}_{checkpoint_path.stem}_{config['decoding']}")
export_dir.mkdir(parents=True, exist_ok=False)
for name, records, loader in [
    ("val", val_records, val_loader),
    ("holdout", holdout_records, holdout_loader),
]:
    predictions, _ = predict(model, loader, device, decoding=config["decoding"], return_logits=True)
    save_predictions(export_dir / f"{name}.jsonl", records, predictions)
    del predictions
    gc.collect()
save_predictions(export_dir / "test.jsonl", test_records, test_predictions)
shutil.copy2(checkpoint_path, export_dir / "checkpoint.pt")
shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
save_json(export_dir / "metadata.json", {
    "checkpoint": "checkpoint.pt", "checkpoint_original_name": checkpoint_path.name,
    "checkpoint_sha256": file_sha256(checkpoint_path), "epoch": selected_epoch,
    "model_config": model_config, "training_config": config, "provenance": provenance,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path), "label_to_id": LABEL_TO_ID,
    "decoding": config["decoding"], "postprocessing": "none",
    "val_rows": len(val_records), "holdout_rows": len(holdout_records),
    "test_rows": len(test_records),
})
print("Saved on Drive:", export_dir)


Predict:   0%|          | 0/985 [00:00<?, ?it/s]

Predict:   0%|          | 0/987 [00:00<?, ?it/s]

Saved on Drive: /content/drive/MyDrive/ner_results/bigru/bigru_ws_20261118T172717Z_9f8dbe85_best_viterbi_lines


In [16]:
display(pd.DataFrame([
    {"file": path.name, "size_MB": round(path.stat().st_size / 1024**2, 2)}
    for path in export_dir.iterdir() if path.is_file()
]))


,file,size_MB
0,val.jsonl,196.76
1,holdout.jsonl,196.90
2,test.jsonl,2.73
3,checkpoint.pt,80.79
4,tokenizer.json,2.46
5,metadata.json,0.00


## 11. Finish W&B


In [17]:
run.summary["submission/selected_checkpoint"] = checkpoint_path.name
run.summary["submission/path"] = str(submission_path)
finish_run(run, best_epoch=selected_epoch, best_score=best_score,
           selection_metric=config["selection_metric"], checkpoint_path=checkpoint_path,
           upload_checkpoint=config["upload_checkpoint"])


epoch,▁▅█
time/epoch_seconds,▁██
train/loss,█▂▁
val_v2/FN,█▂▁
val_v2/FP,▁██
val_v2/TP,▁▇█
val_v2/combined_comp_f1,▁▇█
val_v2/combined_loss,█▁▁
val_v2/precision,█▁▁
val_v2/raw/FN,█▂▁
+12,...
